In [ ]:
# Q1: Key Limitations of traditional MapReduce
#
# 1. Disk-based processing:
#    MapReduce writes intermediate results to disk after every stage,causing heavy disk I/O. This makes data processing slower.
# 2. Slow execution:
#    Frequent reading from and writing to disk increases execution time.It is not suitable for fast data processing.
# 3. Poor support for iterative algorithms:
#    Iterative tasks like machine learning require repeated disk access in MapReduce. This leads to high processing overhead.
# 4. Not suitable for real-time processing:
#    MapReduce is designed mainly for batch processing.It cannot efficiently handle streaming or real-time data.
# 5. Complex programming model:
#    Developers must write separate map and reduce functions.Even simple tasks require more code and effort.
# 6. Limited analytics capabilities:
#    Advanced analytics need additional external tools with MapReduce. It lacks built-in support for machine learning and graph processing.
# 7. Higher latency:
#    Disk-based processing and multiple execution stages increase response time. This results in higher latency for data analysis.
# Why Spark is preferred:
# Spark performs in-memory processing, making it much faster than MapReduce. It also supports real-time processing, machine learning,graph processing, and simpler APIs, making it a better choice for modern big data applications.

In [ ]:
# Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.
# In traditional disk-based systems like MapReduce, every iteration of a machine learning algorithm reads data from disk, processes it,writes results back to disk, then the next iteration reads from disk again.
# For 100 iterations = 100 disk reads + 100 disk writes which is extremely slow.
# Spark solves this by keeping data in RAM (memory) using a concept
# called RDD (Resilient Distributed Dataset) or DataFrame caching.
#
# How it works:
# -> First iteration: data is loaded from disk into memory once
# -> All subsequent iterations: data is read directly from RAM
# -> No disk I/O between iterations = massively faster

In [9]:
import pandas as pd
import numpy as np

# Load E-Commerce Dataset
df = pd.read_csv(r'C:\Users\harsh\Desktop\Celebal Technologies Internship\data.csv', encoding='latin1')

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
df.head()

Shape: (541909, 8)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicates: 5268


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [10]:
# Q3.Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.
print("Before removing duplicates:")
print("Total rows:", df.shape[0])
print("duplicate rows found :" ,df.duplicated().sum())

# Remove duplicates
df_clean = df.drop_duplicates(subset=['InvoiceNo', 'StockCode'])

print("\nAfter removing duplicates:")
print("Total rows:", df_clean.shape[0])

Before removing duplicates:
Total rows: 541909
duplicate rows found : 5268

After removing duplicates:
Total rows: 531225


In [11]:
#Q4:Filter for West region, group by product_category, find average sale_amount
print("Original rows:", df.shape[0])

# Filter United Kingdom + GroupBy Country + Average UnitPrice
df_result = df[df['Country'] == 'United Kingdom'].groupby('Country')['UnitPrice'].mean().reset_index()

df_result.columns = ['Country', 'avg_unit_price']
df_result['avg_unit_price'] = df_result['avg_unit_price'].round(2)

print("Average Unit Price for United Kingdom:")
print(df_result)

Original rows: 541909
Average Unit Price for United Kingdom:
          Country  avg_unit_price
0  United Kingdom            4.53


In [10]:
# Q5: Difference between dropna() and fillna()

#dropna() — removes entire rows that contain null values.Use dropna() when the row is unusable without that value.
#fillna() — replaces null values with a specified value.Use fillna() when you can substitute a sensible default.

#Example:

import numpy as np

data2 = {
    'user_id': [1, 2, 3, 4],
    'status': ['Active', None, 'Inactive', None]
}
df = pd.DataFrame(data2)
print(df)

   user_id    status
0        1    Active
1        2       NaN
2        3  Inactive
3        4       NaN


In [11]:
df['status'] = df['status'].fillna('Unknown')
print("After filling nulls:")
print(df)

After filling nulls:
   user_id    status
0        1    Active
1        2   Unknown
2        3  Inactive
3        4   Unknown


In [12]:
# Q6: Find total count of records for each city

print("Original DataFrame rows:", df.shape[0])

# GroupBy Country + Count + Filter count > 1000
df_result = df.groupby('Country')['InvoiceNo'].count().reset_index()

df_result.columns = ['Country', 'count']
df_result = df_result[df_result['count'] > 1000]

print("Countries with more than 1000 transactions:")
print(df_result)

Original DataFrame rows: 541909
Countries with more than 1000 transactions:
           Country   count
0        Australia    1259
3          Belgium    2069
10            EIRE    8196
13          France    8557
14         Germany    9495
24     Netherlands    2371
25          Norway    1086
27        Portugal    1519
31           Spain    2533
33     Switzerland    2002
36  United Kingdom  495478


In [ ]:
# Q7.Immutability of Spark DataFrames

# Spark DataFrames are immutable once created, they cannot be modified in place. Every transformation like dropping columns or renaming them creates a NEW DataFrame instead of changing the original.

# This means during data cleaning you must always assign the result to a new variable:

# Dropping a column : df_cleaned = df.drop("unwanted_column")

# Renaming a column : df_renamed = df.withColumnRenamed("old_name", "new_name")

In [13]:
# Q8: Filter rows where Quantity is between 1-100
# and Country is 'United Kingdom'
#
# Note: Original question asks for age (18-30) and subscription (Premium)
# but our E-Commerce dataset does not have age/subscription columns.
# Using Quantity and Country as equivalent filters to demonstrate
# the same multi-condition filtering concept.

print("Original rows:", df.shape[0])

df_result = df[(df['Quantity'] >= 1) & (df['Quantity'] <= 100) & (df['Country'] == 'United Kingdom')]

print("Filtered Result:")
print("Total rows:", df_result.shape[0])
print(df_result[['InvoiceNo', 'Description', 'Quantity', 'Country']].head(10))

Original rows: 541909
Filtered Result:
Total rows: 482714
  InvoiceNo                          Description  Quantity         Country
0    536365   WHITE HANGING HEART T-LIGHT HOLDER         6  United Kingdom
1    536365                  WHITE METAL LANTERN         6  United Kingdom
2    536365       CREAM CUPID HEARTS COAT HANGER         8  United Kingdom
3    536365  KNITTED UNION FLAG HOT WATER BOTTLE         6  United Kingdom
4    536365       RED WOOLLY HOTTIE WHITE HEART.         6  United Kingdom
5    536365         SET 7 BABUSHKA NESTING BOXES         2  United Kingdom
6    536365    GLASS STAR FROSTED T-LIGHT HOLDER         6  United Kingdom
7    536366               HAND WARMER UNION JACK         6  United Kingdom
8    536366            HAND WARMER RED POLKA DOT         6  United Kingdom
9    536367        ASSORTED COLOUR BIRD ORNAMENT        32  United Kingdom


Q9: Why handle null values before aggregations?

Pandas/Spark automatically skips nulls in sum() or avg()
without any warning, which gives incorrect results.

Example:

Values: [10, 20, None, 40]

avg() without handling null = (10+20+40)/3 = 23.3  (Wrong)

avg() after fillna(0)       = (10+20+0+40)/4 = 17.5 (Correct)

So always handle nulls first to get accurate results.

Q10.Cast raw_timestamp to datetime and rename to event_time

In [14]:
# Q10.
# Note: Original question asks to cast 'raw_timestamp' column but our E-Commerce dataset has 'InvoiceDate' column which serves the same purpose — a date stored as String type.
# So we cast 'InvoiceDate' to DateTime and rename it to 'event_time' to demonstrate the same concept.

print("Before casting:")
print("InvoiceDate dtype:", df['InvoiceDate'].dtype)
print(df['InvoiceDate'].head())

df['event_time'] = pd.to_datetime(df['InvoiceDate'])
df = df.drop(columns=['InvoiceDate'])

print("After casting and renaming:")
print("event_time dtype:", df['event_time'].dtype)
print(df['event_time'].head())

Before casting:
InvoiceDate dtype: str
0    12/1/2010 8:26
1    12/1/2010 8:26
2    12/1/2010 8:26
3    12/1/2010 8:26
4    12/1/2010 8:26
Name: InvoiceDate, dtype: str
After casting and renaming:
event_time dtype: datetime64[us]
0   2010-12-01 08:26:00
1   2010-12-01 08:26:00
2   2010-12-01 08:26:00
3   2010-12-01 08:26:00
4   2010-12-01 08:26:00
Name: event_time, dtype: datetime64[us]


In [ ]:
# Q11: Shuffle Process in GroupBy — Wide Transformation
#
# When groupBy() runs in Spark, a Shuffle occurs:
#  ->Data is spread across multiple machines in partitions
#  ->Spark moves all rows with the SAME key to the SAME machine
#  ->This movement of data across network is called Shuffle
#
# Why it is a Wide Transformation:
#  Narrow: each partition goes to ONE output partition (filter, select)
#  Wide: data moves across MULTIPLE partitions (groupBy, join)
#
# Shuffling is expensive — minimizing it is a key Spark optimization.

In [15]:
# Q12: Remove rows where email is null OR username is empty string

# Note: Original question asks for 'email' and 'username' columns but our E-Commerce dataset has 'CustomerID' and 'Description' which serve the same purpose for demonstrating the same concept.
# So I am removing rows where CustomerID is null OR Description is empty string

print("Before cleaning:")
print("Total rows:", df.shape[0])
print("CustomerID nulls:", df['CustomerID'].isnull().sum())
print("Description nulls:", df['Description'].isnull().sum())

# Step 1: Remove rows where CustomerID is null
df_clean = df.dropna(subset=['CustomerID'])
print("After dropping null CustomerID:")
print("Total rows:", df_clean.shape[0])

# Step 2: Remove rows where Description is empty string
df_clean = df_clean[df_clean['Description'] != '']
print("After removing empty Description:")
print("Total rows:", df_clean.shape[0])
print("Rows removed:", df.shape[0] - df_clean.shape[0])


Before cleaning:
Total rows: 541909
CustomerID nulls: 135080
Description nulls: 1454
After dropping null CustomerID:
Total rows: 406829
After removing empty Description:
Total rows: 406829
Rows removed: 135080


In [16]:
# Q13: Calculate min, max and mean of price column using agg()
print("UnitPrice Statistics:")
result = df['UnitPrice'].agg(['min', 'max', 'mean'])
print(result)

UnitPrice Statistics:
min    -11062.060000
max     38970.000000
mean        4.611114
Name: UnitPrice, dtype: float64


In [ ]:
# Q14: Risk of using inferSchema=True with messy date formats
# When inferSchema=True, Spark automatically detects data types by scanning the data.
# Risk with messy dates:
# 1. Spark may read date column as String instead of DateType if format is inconsistent
#    e.g. "2024-01-01" mixed with "01/01/2024" or "Jan 1 2024"
# 2. If inferred as String, date operations like filtering by date range or sorting will give wrong results
# 3. Spark may silently infer wrong type without any error
# Better approach: define schema explicitly using StructType so you control what type each column should be, regardless of how messy the source data is.

In [17]:
# Q15: Final processing pipeline
# Note: Original question asks for 'price' and 'store_id' columns but our E-Commerce dataset has 'UnitPrice' and 'Country' columns
# which serve the same purpose:
# 'UnitPrice' is used instead of 'price'
# 'Country' is used instead of 'store_id' for grouping
# The pipeline logic remains exactly the same.

print("Original DataFrame:")
print("Total rows:", df.shape[0])
print("Duplicates:", df.duplicated().sum())
print("Nulls in CustomerID:", df['CustomerID'].isnull().sum())

# Step 1: Remove duplicates
df = df.drop_duplicates()
print("After removing duplicates:")
print("Total rows:", df.shape[0])

# Step 2: Fill null CustomerID with 0
df['CustomerID'] = df['CustomerID'].fillna(0)
print("After filling null CustomerID:")
print("Nulls remaining:", df['CustomerID'].isnull().sum())

# Step 3: Group by Country to calculate total revenue
df_result = df.groupby('Country')['UnitPrice'].sum().reset_index()

df_result.columns = ['Country', 'total_revenue']
df_result = df_result.sort_values('total_revenue', ascending=False)

print("Total Revenue per Country:")
print(df_result.head(10))

Original DataFrame:
Total rows: 541909
Duplicates: 5268
Nulls in CustomerID: 135080
After removing duplicates:
Total rows: 536641
After filling null CustomerID:
Nulls remaining: 0
Total Revenue per Country:
           Country  total_revenue
36  United Kingdom    2233247.604
10            EIRE      48400.370
13          France      42985.980
14         Germany      37633.440
30       Singapore      25108.890
27        Portugal      13010.930
31           Spain      12621.500
16       Hong Kong      12224.400
3          Belgium       7540.130
33     Switzerland       6795.590
